# PFT Flow Execution State Report

Run all cells (**Run All**) to generate a complete state snapshot for a `test_pft_flow` execution.

**Go/no-go criterion**: `pft_test_validation_log` shows **1000/1000** for all 5 data types across all 10 facilities.

Set `EXECUTION_ID` in the next cell, or leave it empty to auto-detect the latest run.

In [ ]:
import psycopg2
import pandas as pd
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ────────────────────────────────────────────────────────────
# Specific execution ID to report on — leave empty to auto-detect the latest.
EXECUTION_ID = ""

# noetl control DB  (execution state, events)
NOETL_HOST     = "722-2.local"
NOETL_PORT     = 54321
NOETL_DBNAME   = "noetl"
NOETL_USER     = "noetl"
NOETL_PASSWORD = "noetl"

# demo_noetl results DB  (work queue, validation log, result tables)
DEMO_HOST     = "722-2.local"
DEMO_PORT     = 54321
DEMO_DBNAME   = "demo_noetl"
DEMO_USER     = "demo"
DEMO_PASSWORD = "demo"
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
conn_noetl = psycopg2.connect(
    host=NOETL_HOST, port=NOETL_PORT,
    dbname=NOETL_DBNAME, user=NOETL_USER, password=NOETL_PASSWORD
)
conn_noetl.autocommit = True

conn_demo = psycopg2.connect(
    host=DEMO_HOST, port=DEMO_PORT,
    dbname=DEMO_DBNAME, user=DEMO_USER, password=DEMO_PASSWORD
)
conn_demo.autocommit = True

print(f"Connected: {NOETL_DBNAME} and {DEMO_DBNAME} @ {NOETL_HOST}:{NOETL_PORT}")


def qn(sql, params=None):
    """Query the noetl control DB."""
    with conn_noetl.cursor() as cur:
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


def qd(sql, params=None):
    """Query the demo_noetl results DB."""
    with conn_demo.cursor() as cur:
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


# Auto-detect latest test_pft_flow execution
if not EXECUTION_ID:
    df_latest = qn("""
        SELECT e.execution_id, c.path, e.status, e.start_time, e.end_time,
               e.last_node_name
        FROM noetl.execution e
        JOIN noetl.catalog c ON c.catalog_id = e.catalog_id
        WHERE c.path ILIKE '%pft_flow%'
          AND e.parent_execution_id IS NULL
        ORDER BY e.created_at DESC
        LIMIT 5
    """)
    print("Recent test_pft_flow executions:")
    display(df_latest)
    if not df_latest.empty:
        EXECUTION_ID = str(df_latest.iloc[0]["execution_id"])
        print(f"\nAuto-selected: {EXECUTION_ID}")
    else:
        print("No pft_flow executions found. Set EXECUTION_ID manually.")
else:
    print(f"Using configured execution: {EXECUTION_ID}")

## Execution Overview

In [ ]:
df_ov = qn("""
    SELECT
        e.execution_id,
        c.path                                               AS playbook_path,
        e.status,
        e.start_time,
        e.end_time,
        COALESCE(
            EXTRACT(EPOCH FROM (e.end_time   - e.start_time)),
            EXTRACT(EPOCH FROM (NOW()        - e.start_time))
        )::int / 60                                          AS duration_min,
        e.last_node_name,
        e.last_event_type,
        e.error
    FROM noetl.execution e
    JOIN noetl.catalog c ON c.catalog_id = e.catalog_id
    WHERE e.execution_id = %s
""", (int(EXECUTION_ID),))

if df_ov.empty:
    print(f"Execution {EXECUTION_ID} not found.")
else:
    display(df_ov.T)
    row = df_ov.iloc[0]
    print(f"\nStatus      : {row['status']}")
    print(f"Last node   : {row['last_node_name']}")
    print(f"Duration    : {row['duration_min']} min")

## Work Queue — Progress by Facility & Data Type

Pivot of `done` counts per (facility × data_type). Target: **1000** in every cell.  
Green = complete, yellow = in progress, grey = not started yet.

In [ ]:
df_q = qd("""
    SELECT
        facility_mapping_id,
        data_type,
        COUNT(*) FILTER (WHERE status = 'pending')  AS pending,
        COUNT(*) FILTER (WHERE status = 'claimed')  AS claimed,
        COUNT(*) FILTER (WHERE status = 'done')     AS done,
        COUNT(*) FILTER (WHERE status = 'error')    AS error,
        COUNT(*)                                     AS total
    FROM public.pft_test_patient_work_queue
    WHERE execution_id = %s
    GROUP BY facility_mapping_id, data_type
    ORDER BY facility_mapping_id, data_type
""", (EXECUTION_ID,))

if df_q.empty:
    print("No work queue rows found.")
else:
    DATA_TYPES = ["assessments", "conditions", "medications", "vital_signs", "demographics"]
    present_types = [t for t in DATA_TYPES if t in df_q["data_type"].unique()]

    pivot_done = df_q.pivot_table(
        index="facility_mapping_id", columns="data_type",
        values="done", aggfunc="sum", fill_value=0
    ).reindex(columns=present_types)

    def color_cell(val):
        if val == 1000:
            return "background-color: #d4edda; color: #155724"
        elif val == 0:
            return "background-color: #f8f9fa; color: #adb5bd"
        else:
            return "background-color: #fff3cd; color: #856404"

    print("Done counts per facility × data_type  (green=1000, yellow=in-progress, grey=0)")
    display(pivot_done.style.applymap(color_cell))

    totals = df_q.groupby("data_type")[["done", "total"]].sum().reindex(present_types)
    totals["pct_done"] = (totals["done"] / totals["total"] * 100).round(1)
    print("\nTotals per data type:")
    display(totals)

## Overall Completion Summary

In [ ]:
df_s = qd("""
    SELECT
        data_type,
        COUNT(*) FILTER (WHERE status = 'done')     AS done,
        COUNT(*) FILTER (WHERE status = 'claimed')  AS claimed,
        COUNT(*) FILTER (WHERE status = 'pending')  AS pending,
        COUNT(*) FILTER (WHERE status = 'error')    AS errors,
        COUNT(*)                                     AS total,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE status = 'done')
            / NULLIF(COUNT(*), 0), 1
        ) AS pct_done
    FROM public.pft_test_patient_work_queue
    WHERE execution_id = %s
    GROUP BY data_type
    ORDER BY data_type
""", (EXECUTION_ID,))

if df_s.empty:
    print("No queue rows found.")
else:
    display(df_s)
    total_done = df_s["done"].sum()
    total_rows = df_s["total"].sum()
    print(f"\nOverall: {total_done:,} / {total_rows:,} ({100 * total_done / total_rows:.1f}% done)")

## Validation Log

Written by `validate_facility_results` after each facility completes.  
All numeric columns should equal **1000** (green) when the run passes.

In [ ]:
df_v = qd("""
    SELECT
        facility_mapping_id,
        assessments_done,
        conditions_done,
        medications_done,
        vital_signs_done,
        demographics_done,
        assessments_queue_done,
        total_expected,
        logged_at
    FROM public.pft_test_validation_log
    WHERE execution_id = %s
    ORDER BY facility_mapping_id
""", (EXECUTION_ID,))

DATA_COLS = [
    "assessments_done", "conditions_done", "medications_done",
    "vital_signs_done", "demographics_done", "assessments_queue_done",
]

if df_v.empty:
    print("No validation log entries yet — facilities not yet validated.")
else:
    def highlight_val(val):
        if pd.isna(val):
            return "background-color: #fff3cd"
        return (
            "background-color: #d4edda; color: #155724" if int(val) == 1000
            else "background-color: #f8d7da; color: #721c24"
        )

    present = [c for c in DATA_COLS if c in df_v.columns]
    display(df_v.style.applymap(highlight_val, subset=present))

    n_done = len(df_v)
    all_pass = all(df_v[c].fillna(0).astype(int).eq(1000).all() for c in present)

    print(f"\nFacilities validated: {n_done} / 10")
    if all_pass and n_done == 10:
        print("GO — all 10 facilities, all data types: 1000/1000")
    else:
        print("NO-GO — still in progress or shortfalls present")
        shortfalls = [
            f"  {c}: facilities {list(df_v.loc[df_v[c].fillna(0).astype(int) != 1000, 'facility_mapping_id'])}"
            for c in present
            if not df_v[c].fillna(0).astype(int).eq(1000).all()
        ]
        for s in shortfalls:
            print(s)

## Result Table Row Counts

Spot-check that rows are accumulating in the downstream result tables.

In [ ]:
RESULT_TABLES = [
    "pft_test_patient_assessments",
    "pft_test_patient_conditions",
    "pft_test_patient_medications",
    "pft_test_patient_vital_signs",
    "pft_test_patient_demographics",
    "pft_test_mds_assessment_ids_work",
    "pft_test_mds_assessment_details",
]

rows = [
    {"table": t, "row_count": qd(f"SELECT COUNT(*) AS cnt FROM public.{t}")["cnt"].iloc[0]}
    for t in RESULT_TABLES
]
display(pd.DataFrame(rows))

## Recent NoETL Events (last 50)

Useful for spotting stuck steps, errors, or loop anomalies.

In [ ]:
df_ev = qn("""
    SELECT
        event_id,
        event_type,
        node_name,
        node_type,
        status,
        ROUND(duration::numeric, 3) AS duration_s,
        created_at,
        error
    FROM noetl.event
    WHERE execution_id = %s
    ORDER BY event_id DESC
    LIMIT 50
""", (int(EXECUTION_ID),))

print(f"Last {len(df_ev)} events:")
display(df_ev)

## Step (Node) Completion Counts

Event counts grouped by `node_name` + `event_type` — spot over- or under-firing steps.

In [ ]:
df_sc = qn("""
    SELECT
        node_name,
        event_type,
        COUNT(*)                         AS count,
        MIN(created_at)                  AS first_seen,
        MAX(created_at)                  AS last_seen,
        ROUND(AVG(duration)::numeric, 3) AS avg_duration_s
    FROM noetl.event
    WHERE execution_id = %s
    GROUP BY node_name, event_type
    ORDER BY node_name, event_type
""", (int(EXECUTION_ID),))

print(f"Node event summary ({len(df_sc)} rows):")
display(df_sc)

## Close Connections

In [ ]:
conn_noetl.close()
conn_demo.close()
print("Connections closed.")